# SECTION - 1 
# LOAD THE DATA, BASIC INSPECTION

In [1]:
import pandas as pd
import numpy as np

In [39]:
df = pd.read_excel('kinara_kirana_loyalty_raw.xlsx')

In [40]:
# shape
print("Shape:", df.shape)


print("\nDtypes:")
print(df.dtypes)


df.head()

Shape: (2374, 21)

Dtypes:
member_id                 object
join_date                 object
tier                      object
city                      object
age                        int64
gender                    object
acquisition_channel       object
preferred_channel         object
transaction_count          int64
total_spend                int64
avg_basket_value         float64
total_points_earned        int64
total_points_redeemed      int64
last_transaction_date     object
coupon_redemptions       float64
referral_count           float64
membership_status         object
preferred_category        object
app_installed               bool
email_opt_in                bool
sms_opt_in                  bool
dtype: object


,member_id,join_date,tier,city,age,gender,acquisition_channel,preferred_channel,transaction_count,total_spend,...,total_points_earned,total_points_redeemed,last_transaction_date,coupon_redemptions,referral_count,membership_status,preferred_category,app_installed,email_opt_in,sms_opt_in
0,UM100001,2024-03-15,Silver,Bengaluru,23,Female,Campaign,Offline,1,8168,...,816,0,2026-06-30,3.0,0.0,Active,Kids,True,True,False
1,UM100002,2023-02-24,Gold,Chandigarh,44,Male,Referral,Offline,15,62999,...,6299,0,2026-02-03,9.0,2.0,Churn Risk,Electronics,True,True,True
2,UM100003,2026-01-24,Silver,NaN,47,Female,Website,Offline,1,34533,...,3453,1631,2026-06-02,2.0,5.0,Active,Beauty,True,True,True
3,UM100004,2023-11-25,Silver,Kochi,49,Female,Store,Omnichannel,5,9737,...,973,0,2026-06-02,1.0,4.0,Active,Sports,True,True,True
4,UM100005,2025-11-22,Silver,Mumbai,70,Female,Website,Omnichannel,5,45525,...,4552,1723,2026-01-08,6.0,5.0,Dormant,Sports,True,True,True


In [41]:
print("Null Values:")
print(df.isnull().sum())

Null Values:
member_id                 0
join_date                 0
tier                      0
city                      8
age                       0
gender                    0
acquisition_channel       0
preferred_channel         0
transaction_count         0
total_spend               0
avg_basket_value          0
total_points_earned       0
total_points_redeemed     0
last_transaction_date     0
coupon_redemptions       55
referral_count           30
membership_status         0
preferred_category        0
app_installed             0
email_opt_in              0
sms_opt_in                0
dtype: int64


# SECTION 2 
# INVESTIGATING DATA TO FIND INCONSISTENCIES, LOGICAL ERRORS, OR DATA ENTRY ERRORS

In [42]:
df.columns

Index(['member_id', 'join_date', 'tier', 'city', 'age', 'gender',
       'acquisition_channel', 'preferred_channel', 'transaction_count',
       'total_spend', 'avg_basket_value', 'total_points_earned',
       'total_points_redeemed', 'last_transaction_date', 'coupon_redemptions',
       'referral_count', 'membership_status', 'preferred_category',
       'app_installed', 'email_opt_in', 'sms_opt_in'],
      dtype='object')

In [43]:
# unique values in categorical columns - this is where spelling issues will surface
spell_checks = ["tier", "city", "gender", "membership_status", 
            "acquisition_channel", "preferred_channel", 
            "preferred_category", "app_installed", 
            "email_opt_in", "sms_opt_in"]
for col in spell_checks :
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- tier ---
tier
Silver      1678
Gold         519
Platinum     132
Silvr         14
silver        10
SILVER         6
Platnium       5
gold           3
GOLD           3
Golld          3
PLATINUM       1
Name: count, dtype: int64

--- city ---
city
Pune           222
Kochi          209
Mumbai         199
Kolkata        197
Chennai        192
Ahmedabad      189
Delhi          189
Hyderabad      187
Jaipur         184
Bengaluru      183
Lucknow        182
Chandigarh     176
Hyderbad        10
NaN              8
Jaipur           8
Chandigarh       5
Kolkata          5
Ahmedabad        5
Bengalru         3
Pune             3
Kochi            3
Lucknow          3
 Delhi           3
Mumbi            2
Mumabi           2
Bangalore        2
Delhi            1
Banglore         1
Chennai          1
Name: count, dtype: int64

--- gender ---
gender
Male      831
Female    786
Other     757
Name: count, dtype: int64

--- membership_status ---
membership_status
Active        1346
Dormant        69

In [44]:
# duplicate member_id check
print("Total rows:", len(df))
print("Unique member_ids:", df["member_id"].nunique())
print("Duplicate member_ids:", df["member_id"].duplicated().sum())

Total rows: 2374
Unique member_ids: 2374
Duplicate member_ids: 0


In [45]:
print("Age min:", df["age"].min())
print("Age max:", df["age"].max())
print("\nSuspicious ages (below 18 or above 75):")
print(df[(df["age"] < 18) | (df["age"] > 75)][["member_id", "age"]]) # i dont think anything suspicious here, we can let it be as it is

Age min: 16
Age max: 99

Suspicious ages (below 18 or above 75):
     member_id  age
206   UM100207   16
281   UM100282   99
422   UM100423   99
898   UM100899   99
1161  UM101162   99
1654  UM101655   99
1799  UM101800   91
2073  UM102074   16


In [46]:
df["join_date"] = pd.to_datetime(df["join_date"], errors="coerce")
df["last_transaction_date"] = pd.to_datetime(df["last_transaction_date"], errors="coerce")

reference_date = pd.Timestamp("2025-06-30") # to check days passed since last transaction

In [47]:
print("Future last_transaction_date rows:")
print(df[df["last_transaction_date"] > reference_date][["member_id", "last_transaction_date"]])

# join date after last transaction - also suspicious
print("\nRows where join_date is after last_transaction_date:")
print((df[df["join_date"] > df["last_transaction_date"]][["member_id", "join_date", "last_transaction_date"]]).count())


Future last_transaction_date rows:
     member_id last_transaction_date
0     UM100001            2026-06-30
1     UM100002            2026-02-03
2     UM100003            2026-06-02
3     UM100004            2026-06-02
4     UM100005            2026-01-08
...        ...                   ...
2369  UM102370            2025-11-16
2370  UM102371            2026-06-04
2371  UM102372            2026-03-18
2372  UM102373            2026-05-14
2373  UM102374            2026-06-17

[2374 rows x 2 columns]

Rows where join_date is after last_transaction_date:
member_id                55
join_date                55
last_transaction_date    55
dtype: int64


In [48]:
# negative spend
print("Negative total_spend rows:")
print(df[df["total_spend"] < 0][["member_id", "total_spend"]])

Negative total_spend rows:
     member_id  total_spend
345   UM100346       -15702
1614  UM101615       -47164
1911  UM101912       -30259


In [49]:
# business rule: redeemed cannot exceed earned
print("Rows where points_redeemed > points_earned:")
print(df[df["total_points_redeemed"] > df["total_points_earned"]][["member_id", "total_points_earned", "total_points_redeemed"]])

Rows where points_redeemed > points_earned:
     member_id  total_points_earned  total_points_redeemed
382   UM100383                22543                  22893
849   UM100850                23465                  23835
1047  UM101048                 3212                   3993
1205  UM101206                 4448                   5006
1337  UM101338                15948                  16641
2003  UM102004                 2872                   3463
2109  UM102110                 1848                   2485


In [100]:
# outlier inspection using IQR
for col in ["total_spend", "avg_basket_value", "transaction_count"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    outliers = df[df[col] > upper]
    print(outliers[["member_id", "tier", col]].head())

   member_id      tier  total_spend
8   UM100010  Platinum       262108
35  UM100037  Platinum       391683
42  UM100044  Platinum       423905
60  UM100062  Platinum       423449
70  UM100073  Platinum       346741
    member_id    tier  avg_basket_value
6    UM100008  Silver           54289.0
64   UM100067  Silver           51129.0
89   UM100092  Silver           53546.0
140  UM100144  Silver           44238.0
143  UM100147  Silver           58054.0
    member_id      tier  transaction_count
8    UM100010  Platinum                 59
35   UM100037  Platinum                 48
42   UM100044  Platinum                 39
60   UM100062  Platinum                 51
138  UM100142  Platinum                 44


In [52]:
df.shape

(2374, 21)

# SECTION 3
# FIXING THE INCONSISTENCIES AND ERRORS

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2374 entries, 0 to 2373
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   member_id              2374 non-null   object        
 1   join_date              2374 non-null   datetime64[ns]
 2   tier                   2374 non-null   object        
 3   city                   2366 non-null   object        
 4   age                    2374 non-null   int64         
 5   gender                 2374 non-null   object        
 6   acquisition_channel    2374 non-null   object        
 7   preferred_channel      2374 non-null   object        
 8   transaction_count      2374 non-null   int64         
 9   total_spend            2374 non-null   int64         
 10  avg_basket_value       2374 non-null   float64       
 11  total_points_earned    2374 non-null   int64         
 12  total_points_redeemed  2374 non-null   int64         
 13  las

In [73]:
df['last_transaction_date'].value_counts()

last_transaction_date
2026-06-15    61
2026-06-03    61
2026-06-07    57
2026-06-22    54
2026-06-05    53
              ..
2025-07-20     1
2026-03-01     1
2025-09-29     1
2025-11-22     1
2025-11-16     1
Name: count, Length: 288, dtype: int64

In [54]:
# remove leading/trailing spaces from string columns

string_columns = [
    "member_id",
    "city",
    "tier",
    "preferred_channel",
    "gender",
    "acquisition_channel",
    "membership_status",
    "preferred_category"
]

for col in string_columns:
    df[col] = df[col].str.strip()

In [55]:
#city standardization
city_map = {
    "Banglore"  : "Bengaluru",
    "Bengalru"  : "Bengaluru",
    "Bangalore" : "Bengaluru",
    "Mumabi"    : "Mumbai",
    "Mumbi"     : "Mumbai",
    "Hyderbad"  : "Hyderabad",
}
df["city"] = df["city"].replace(city_map)

In [56]:
# dropping all null city roews
df = df.dropna(subset=["city"]).reset_index(drop=True)

In [102]:
df["tier"] = df["tier"].str.strip().str.title()
tier_map = {
    "Silvr"  : "Silver",
    "Golld"  : "Gold",
    "Platnum": "Platinum",
    "Platnium": "Platinum"
}
df["tier"] = df["tier"].replace(tier_map)
print("Tier distribution after fix:")
print(df["tier"].value_counts())

Tier distribution after fix:
tier
Silver      1666
Gold         518
Platinum     136
Name: count, dtype: int64


In [76]:
# fix join_date after last_transaction_date
# these are data entry errors - member exists, transaction exists
# setting last_transaction_date to reference date as conservative fix
reference_date = pd.Timestamp("2026-06-30")

err = df["join_date"] > df["last_transaction_date"]
df.loc[err, "last_transaction_date"] = reference_date

In [77]:
err.sum() #rows where the exceeding transaction dates were fixed

np.int64(55)

In [78]:
df_backup = df.copy()

In [79]:
df = df_backup.copy()

In [83]:
df.shape

(2320, 21)

In [81]:
#remove future dates (after reference date)
df = df[df["last_transaction_date"] <= reference_date].reset_index(drop=True)

In [84]:
df["total_spend"] = df["total_spend"].abs() #to fix negative amount spends

In [85]:
# points redeemed > points earned
# likely historical carry-over, cap redeemed at earned
incons = df["total_points_redeemed"] > df["total_points_earned"]
df.loc[incons, "total_points_redeemed"] = df.loc[incons, "total_points_earned"]

In [86]:
df

,member_id,join_date,tier,city,age,gender,acquisition_channel,preferred_channel,transaction_count,total_spend,...,total_points_earned,total_points_redeemed,last_transaction_date,coupon_redemptions,referral_count,membership_status,preferred_category,app_installed,email_opt_in,sms_opt_in
0,UM100001,2024-03-15,Silver,Bengaluru,23,Female,Campaign,Offline,1,8168,...,816,0,2026-06-30,3.0,0.0,Active,Kids,True,True,False
1,UM100002,2023-02-24,Gold,Chandigarh,44,Male,Referral,Offline,15,62999,...,6299,0,2026-02-03,9.0,2.0,Churn Risk,Electronics,True,True,True
2,UM100004,2023-11-25,Silver,Kochi,49,Female,Store,Omnichannel,5,9737,...,973,0,2026-06-02,1.0,4.0,Active,Sports,True,True,True
3,UM100005,2025-11-22,Silver,Mumbai,70,Female,Website,Omnichannel,5,45525,...,4552,1723,2026-01-08,6.0,5.0,Dormant,Sports,True,True,True
4,UM100006,2026-02-01,Silver,Ahmedabad,43,Other,Website,Offline,4,10476,...,1047,612,2026-06-17,6.0,4.0,Active,Home & Kitchen,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,UM102370,2023-07-12,Platinum,Kolkata,55,Male,QR Scan,Offline,27,254758,...,25475,21789,2025-11-16,6.0,3.0,Churn Risk,Kids,True,True,False
2316,UM102371,2024-11-05,Silver,Lucknow,45,Female,Website,Online,7,52248,...,5224,3931,2026-06-04,10.0,5.0,Active,Electronics,True,True,True
2317,UM102372,2023-02-20,Gold,Mumbai,63,Female,QR Scan,Omnichannel,14,136555,...,13655,5372,2026-03-18,10.0,2.0,Dormant,Beauty,True,False,False
2318,UM102373,2026-05-11,Silver,Hyderabad,74,Other,Website,Omnichannel,4,20361,...,2036,0,2026-05-14,2.0,4.0,Dormant,Home & Kitchen,False,True,True


In [33]:
# null imputation
df["coupon_redemptions"] = df["coupon_redemptions"].fillna(0)
df["referral_count"]     = df["referral_count"].fillna(0)

# opt-in nulls - treat missing as False
for col in ["app_installed", "email_opt_in", "sms_opt_in"]:
    df[col] = df[col].fillna(False)


In [35]:
#flaggin outliers
for col in ["total_spend", "avg_basket_value"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR #using 3 to not flag moderately high values, only flags very high values - also not using 
                         #lower bounds as not interested in very low spends

    flags = []

    for value in df[col]:
        if value > upper:
            flags.append(1)
        else:
            flags.append(0)

    df[col + "_outlier_flag"] = flags

    print(col, "outliers flagged:", sum(flags))

total_spend outliers flagged: 6
avg_basket_value outliers flagged: 3


# Data Quality Assessment — Findings and Treatment Decisions


After running an initial inspection of the UrbanMart loyalty dataset (2,374 rows, 21 columns),
the following inconsistencies were identified. Each issue is documented along with the
treatment decision and reasoning.

---

## 1. City Name Inconsistencies
**Found:** Multiple spelling variants for the same city.
Examples — `Banglore`, `Bengalru`, `Bangalore` for Bengaluru;
`Mumabi`, `Mumbi` for Mumbai; `Hyderbad` for Hyderabad.
Also found 8 rows with null city values.

**Fix:** Replaced all variants using a standardized city lookup map.
Rows with null city values were dropped (8 rows, <0.4% of dataset — negligible).

**Reasoning:** City is a core dimension for geographic segmentation.
Inconsistent naming would cause the same city to appear as multiple
entries in any GROUP BY analysis, splitting revenue and member counts incorrectly.

---

## 2. Tier Name Inconsistencies
**Found:** Mixed casing and typos across the tier column.
Examples — `silver`, `SILVER`, `Silvr` for Silver;
`gold`, `GOLD`, `Golll` for Gold.

**Fix:** Applied `.str.title()` to normalize casing, followed by a
lookup map to correct typos.

**Reasoning:** Tier is the primary segmentation variable in a loyalty program.
Any case mismatch would break tier-level aggregations entirely.

---

## 3. Leading and Trailing Whitespace
**Found:** Whitespace detected across multiple string columns including
city, tier, and membership_status.

**Fix:** Applied `.str.strip()` across all string columns as the first
cleaning step before any other transformation.

**Reasoning:** Whitespace is invisible but causes exact-match failures
in grouping, filtering, and joining operations.

---

## 4. Join Date After Last Transaction Date (55 rows)
**Found:** 55 members had a `join_date` that was more recent than their
`last_transaction_date` — which is logically impossible.

**Fix:** For these rows, `last_transaction_date` was set to the reference
date (2025-06-30) rather than dropping the records.

**Reasoning:** These members have valid spend, points, and behavioral data.
Dropping them would mean losing ~2.3% of the dataset for what is likely
a data entry error on the date field. Setting the transaction date to the
reference date is a conservative fix — it treats these members as recently
active, which is the safest assumption given that all other fields are intact.

---

## 5. Null Values in Coupon Redemptions (55 rows) and Referral Count (30 rows)
**Found:** `coupon_redemptions` had 55 nulls; `referral_count` had 30 nulls.

**Fix:** Both columns were filled with 0.

**Reasoning:** In a loyalty program context, a null in coupon redemptions
most likely means the member never redeemed a coupon, not that the data
is missing. Same logic applies to referral count — absence of a referral
record means no referrals were made. Filling with 0 is more accurate than
imputing with the median, which would artificially inflate engagement signals.

---

## 6. Negative Spend Values (3 rows)
**Found:** 3 rows with negative `total_spend` values
(-15,702, -47,164, -30,259).

**Fix:** Converted to absolute values using `abs()`.

**Reasoning:** Negative spend is not analytically possible for a retail
loyalty program. These are almost certainly data entry errors where a
minus sign was incorrectly included. The magnitude of the values is
consistent with legitimate high-value purchases, so the transaction is
real — only the sign is wrong.

---

## 7. Points Redeemed Exceeding Points Earned (7 rows)
**Found:** 7 members had `total_points_redeemed` greater than
`total_points_earned` in the current dataset.

**Fix:** Capped `total_points_redeemed` at `total_points_earned` for
these rows.

**Reasoning:** This is likely a carry-over issue — members may have
redeemed points earned in a prior period not captured in this dataset
window. Since we cannot recover the historical data, capping at earned
is the most conservative and analytically defensible fix. It prevents
a negative unredeemed points balance from corrupting downstream
redemption rate calculations.

---

## 8. Outliers in Spend and Basket Value
**Found:** 140 members with `total_spend` above the upper IQR fence
(₹2,49,153); 70 members with `avg_basket_value` above ₹43,910.
Notable example — a Platinum member with spend of ₹4,23,905 and
transaction count of 39.

**Fix:** Flagged with `total_spend_outlier_flag` and
`avg_basket_value_outlier_flag` columns. Records were retained.

**Reasoning:** These are likely genuine high-value customers, particularly
at the Platinum tier. Removing them would distort tier-level spend analysis.
Flagging allows downstream filtering without permanent data loss.

---

## Summary of Data Treatment

| Issue | Rows Affected | Treatment |
|---|---|---|
| City spelling variants | ~30 rows | Lookup map standardization |
| City nulls | 8 rows | Dropped |
| Tier inconsistencies | ~50 rows | Title case + lookup map |
| Whitespace | Multiple columns | str.strip() on all string cols |
| Join date after transaction | 55 rows | last_transaction_date set to reference date |
| Coupon redemptions null | 55 rows | Filled with 0 |
| Referral count null | 30 rows | Filled with 0 |
| Negative spend | 3 rows | Converted to absolute value |
| Points redeemed > earned | 7 rows | Capped at earned value |
| Outliers in spend/basket | 140 / 70 rows | Flagged, retained |

# SECTION 4
# FEATURE ENGINEERING - CONSTRUCTING NEW COLUMNS FROM ALREADY EXISITING ONES FOR BETTER UNDERSTANDING AND ANALYSIS

In [115]:
# days since last transaction
# this tells us how recently a member was active
# TO CHECK IF CUSTOMER CHURNED

reference_date = pd.Timestamp("2026-06-30")

days_since = []
for date in df["last_transaction_date"]:
    diff = (reference_date - date).days
    days_since.append(diff)

df["days_since_last_txn"] = days_since

In [116]:
#CHURN SIGNAL, ASSUMED THAT CUSTOMER CHURNED IF NO TRANSACTION IN LAST 90 DAYS
churn_signals = []
for days in df["days_since_last_txn"]:
    if days > 90:
        churn_signals.append(1)
    else:
        churn_signals.append(0)

df["churn_signal"] = churn_signals

In [114]:
df.drop(columns = ['days_since_last_txn', 'churn_signal'], inplace  = True)

In [117]:
df.head()

,member_id,join_date,tier,city,age,gender,acquisition_channel,preferred_channel,transaction_count,total_spend,...,referral_count,membership_status,preferred_category,app_installed,email_opt_in,sms_opt_in,points_unredeemed,high_value_flag,days_since_last_txn,churn_signal
0,UM100001,2024-03-15,Silver,Bengaluru,23,Female,Campaign,Offline,1,8168,...,0.0,Active,Kids,True,True,False,816,0,0,0
1,UM100002,2023-02-24,Gold,Chandigarh,44,Male,Referral,Offline,15,62999,...,2.0,Churn Risk,Electronics,True,True,True,6299,0,147,1
2,UM100004,2023-11-25,Silver,Kochi,49,Female,Store,Omnichannel,5,9737,...,4.0,Active,Sports,True,True,True,973,0,28,0
3,UM100005,2025-11-22,Silver,Mumbai,70,Female,Website,Omnichannel,5,45525,...,5.0,Dormant,Sports,True,True,True,2829,0,173,1
4,UM100006,2026-02-01,Silver,Ahmedabad,43,Other,Website,Offline,4,10476,...,4.0,Active,Home & Kitchen,False,False,False,435,0,13,0


In [88]:
# unredeemed points
# points earned but never used
# this is a liability for the brand - they owe these points to customers

df["points_unredeemed"] = df["total_points_earned"] - df["total_points_redeemed"]

In [89]:
# FINDING TOP 20% CUSTOMERS - HIGH SPEND GUYS

top_20_threshold = df["avg_basket_value"].quantile(0.80)

high_value_flags = []
for value in df["avg_basket_value"]:
    if value >= top_20_threshold:
        high_value_flags.append(1)
    else:
        high_value_flags.append(0)

df["high_value_flag"] = high_value_flags

In [90]:
df["high_value_flag"].value_counts() 

high_value_flag
0    1856
1     464
Name: count, dtype: int64

In [92]:
grouped = df.groupby("high_value_flag").agg({
    "avg_basket_value": "sum"
})

grouped["contribution_percent"] = (
    grouped["avg_basket_value"] /
    grouped["avg_basket_value"].sum()
) * 100

In [94]:
grouped #20% customers contributed to more than 50% of revenue, again this was for checking not making new features, hehe

,avg_basket_value,contribution_percent
high_value_flag,,
0,13371528.36,49.686951
1,13540020.93,50.313049


# SECTION 5
# EXPORTING DIRECTLY TO POSTGRESQL

In [96]:

from sqlalchemy import create_engine

engine = create_engine('postgresql://postgres:@localhost:5432/postgres')

# Test the connection first
with engine.connect() as conn:
    print("Connection successful")

Connection successful


In [97]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execution_options(isolation_level="AUTOCOMMIT").execute(
        text("CREATE DATABASE kinara_kirana")
    )
    print("Database created")

Database created


In [118]:
engine_taiwan = create_engine('postgresql://postgres:@localhost:5432/taiwan_credit')

df.to_sql('kinara_kirana_loyalty', engine_taiwan, if_exists='replace', index=False)
print("table loaded")

table loaded


In [101]:
df.columns

Index(['member_id', 'join_date', 'tier', 'city', 'age', 'gender',
       'acquisition_channel', 'preferred_channel', 'transaction_count',
       'total_spend', 'avg_basket_value', 'total_points_earned',
       'total_points_redeemed', 'last_transaction_date', 'coupon_redemptions',
       'referral_count', 'membership_status', 'preferred_category',
       'app_installed', 'email_opt_in', 'sms_opt_in', 'days_since_last_txn',
       'points_unredeemed', 'high_value_flag'],
      dtype='object')

## JUST DID ONE MISTAKE, LEFT NULL VALUES IN COUPON REDEMPTIONS AND REFERRAL COUNTS, COULD HAVE SOLVED IT IN PANDAS AGAIN  